# 06 · Sequential invoice workflow

SequentialAgent runs children in order. output_key saves one stage's result, and instruction placeholders feed it into the next stage.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 4 Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import SequentialAgent
lookup = llm("lookup", "Use get_order to retrieve O1001. Report only the returned facts.", tools=[get_order], output_key="order_facts")
draft = llm("draft", "Write a brief invoice email using these facts: {order_facts}. Do not invent payment status.", output_key="email_draft")
review = llm("review", "Check this draft against the order facts and return a corrected short email. "
             "Facts: {order_facts}. Draft: {email_draft}.")
workflow = SequentialAgent(name="invoice_pipeline", sub_agents=[lookup, draft, review])
lab = await make_lab(workflow)
await ask(lab, "Prepare an invoice summary email for O1001; do not send it.")
print("State keys:", list(lab.session.state))


## Try it

Add a fourth agent that generates a subject line. Always construct fresh child agents when building another workflow.


## Reference

[Official ADK documentation](https://adk.dev/agents/workflow-agents/sequential-agents/). Shared configuration: `config.json`. No environment activation or login cells are needed.
